![pyFlir](pyflir_logo.png)

# pyFlir Showcase

Full workflow for a FLIR GigE Vision thermal camera:
discovery, XML download, configuration, single-frame grab, batch acquisition, video recording, and visualization.

See the [README](README.md) for installation and API reference.

## Setup

In [ ]:
# Firewall: allow inbound UDP for GVSP streaming (run once, needs admin — Windows only)
# import sys, subprocess
# exe = sys.executable
# result = subprocess.run(
#     ["netsh", "advfirewall", "firewall", "show", "rule", "name=pyFlir-GVSP"],
#     capture_output=True, text=True)
# if "No rules match" in result.stderr or result.returncode != 0:
#     print(f"Adding firewall rule for: {exe}")
#     subprocess.run(
#         ["netsh", "advfirewall", "firewall", "add", "rule",
#          "name=pyFlir-GVSP", "dir=in", "action=allow",
#          "protocol=UDP", f"program={exe}"],
#         capture_output=True, text=True)
# else:
#     print("Firewall rule already exists.")

# Linux: increase UDP receive buffer (run once, needs sudo)
# import subprocess
# subprocess.run(["sudo", "sysctl", "-w", "net.core.rmem_max=16777216"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from pyflir import Camera, discover

## 1. Discovery

Sends a GVCP broadcast on every host network interface and lists all
responding GigE Vision cameras. Works regardless of camera IP or subnet.

In [ ]:
cameras = discover()
for c in cameras:
    serial = c.get('serial', '?')
    print(f"{c.get('manufacturer','?')} {c.get('model','?')}  "
          f"ip={c['ip']}  serial={serial}")

## 2. Connect & download XML

FLIR cameras describe all of their features in a GenICam XML file stored
on the camera. Download it once and save it locally; afterwards load it
from disk to avoid the network round-trip on every reconnect.

In [ ]:
cam = Camera()      # auto-discover first camera; or Camera(ip="169.254.x.x")
cam.connect()
print(f"Connected  @  {cam.ip}  (model and serial available after load_xml)")

In [ ]:
import os, glob

# Download the GenICam XML from the camera (skip if already cached locally)
xml_files = glob.glob("docs/camera_*.xml") + glob.glob("camera_*.xml")
if xml_files:
    xml_file = xml_files[0]
    print(f"Using cached XML: {xml_file}")
else:
    cam.download_xml()
    xml_file = glob.glob("camera_*.xml")[0]
    os.makedirs("docs", exist_ok=True)
    import shutil; shutil.copy(xml_file, f"docs/{os.path.basename(xml_file)}")
    print(f"Downloaded and saved: docs/{os.path.basename(xml_file)}")

cam.load_xml(xml_file)
print(f"Model  : {cam.model}")
print(f"Serial : {cam.serial}")
print(f"Image  : {cam.width} × {cam.height} px")
print(f"Nodes  : {len(cam._nodes)} register nodes loaded")

## 3. Configure

### Frame rate and exposure

In [ ]:
cam.exposure_ms = 80.0       # integration time in milliseconds
cam.frame_rate  = 30.0      # Hz — check frame_rate_max first if unsure

print(f"Exposure:       {cam.exposure_ms:.2f} ms")
print(f"Frame rate:     {cam.frame_rate:.1f} Hz")
if cam.frame_rate_max:
    print(f"Max frame rate: {cam.frame_rate_max:.1f} Hz  (for current ROI)")
print(f"FPA temperature:{cam.detector_temperature:.1f} °C")

### Calibration block

FLIR cameras store multiple calibration presets, each covering a different
temperature range. Select the block that matches your scene.

In [ ]:
blocks = cam.get_calibration_blocks()
print(f"{'Index':>5}  {'Name':<30}  {'Range (°C)':>14}  Lens")
print("-" * 70)
for b in blocks:
    active = " ◀ active" if b['index'] == cam.get_calibration_block() else ""
    print(f"{b['index']:>5}  {b.get('name',''):<30}  "
          f"{b['tmin']:>5.0f} – {b['tmax']:<5.0f}  "
          f"{b.get('lens','')} {active}")

In [ ]:
# Select block 0 (coldest range)
cam.set_calibration_block(0)
print(f"Active block: {cam.get_calibration_block()}")

### Radiometric object parameters

These affect the accuracy of temperature measurements. Set them to match
your measurement scenario.

In [ ]:
cam.set_object_params(
    emissivity  = 0.95,     # surface emissivity (0–1)
    distance_m  = 1.0,      # object-to-camera distance in metres
    atm_temp_K  = 293.15,   # atmospheric temperature in Kelvin (20 °C)
    refl_temp_K = 293.15,   # reflected temperature in Kelvin
    humidity    = 0.50,     # relative humidity (0–1)
)
params = cam.get_object_params()
for k, v in params.items():
    print(f"  {k}: {v}")

### ROI (optional)

Reduce the acquisition window for higher frame rates.
Width and height must be multiples of the camera's increment constraints.

In [ ]:
limits = cam.get_roi_limits()
print("ROI limits:", limits)
print("Current ROI:", cam.get_roi())

# Example: switch to a smaller window
# cam.set_roi(256, 256)
# cam.frame_rate = cam.frame_rate_max
# print(f"New ROI: {cam.get_roi()}, max fps: {cam.frame_rate_max:.1f} Hz")

## 4. Grab a single frame

Starts the stream, captures one frame, and stops. Frames are 16-bit raw
digital counts (uint16). Use percentile clipping to handle warm-spot outliers.

In [ ]:
frame = cam.grab()
print(f"Shape: {frame.shape}, dtype: {frame.dtype}, "
      f"min={frame.min()}, max={frame.max()}, mean={frame.mean():.0f}")

vmin, vmax = np.percentile(frame, [1, 99])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(frame, cmap='inferno', vmin=vmin, vmax=vmax)
ax.set_title(f'Single frame  {frame.shape[1]}×{frame.shape[0]} px')
plt.colorbar(im, ax=ax, label='Raw digital counts (uint16)')
plt.tight_layout()
plt.show()

## 5. Acquire multiple frames

Streams N frames and returns them as a list of 2-D arrays.
Stack into a 3-D numpy array `(N, H, W)` for analysis.

In [ ]:
frames = cam.acquire(20)
data   = np.stack(frames)            # (N, H, W) uint16
print(f"Acquired: {data.shape}, dtype: {data.dtype}, size: {data.nbytes / 1e6:.1f} MB")

In [ ]:
n, h, w = data.shape
fps     = cam.frame_rate
vmin, vmax = np.percentile(data, [1, 99])

indices = np.linspace(0, n - 1, 3, dtype=int)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, idx in zip(axes, indices):
    im = ax.imshow(data[idx], cmap='inferno', vmin=vmin, vmax=vmax)
    ax.set_title(f'Frame {idx}  (t = {idx / fps:.3f} s)')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 6. Record a video

Stream at the desired frame rate, capture all frames, and save to an `.npy`
file. The `npy` format preserves the raw uint16 values without any lossy
compression — useful for quantitative analysis.

Playback is shown inline as an animated figure.

In [ ]:
# --- recording settings ---
RECORD_FPS      = 25      # Hz  (must be ≤ frame_rate_max)
RECORD_DURATION = 3.0     # seconds
N_FRAMES        = int(RECORD_FPS * RECORD_DURATION)
OUTPUT_FILE     = "recording.npy"

cam.frame_rate = RECORD_FPS
print(f"Recording {N_FRAMES} frames at {RECORD_FPS} fps  "
      f"({RECORD_DURATION:.1f} s)  …")

video = cam.acquire(N_FRAMES, timeout=RECORD_DURATION + 10.0)
video = np.stack(video)                   # (N, H, W) uint16

np.save(OUTPUT_FILE, video)
print(f"Saved {video.shape}  {video.dtype}  →  {OUTPUT_FILE}  "
      f"({video.nbytes / 1e6:.1f} MB)")

In [ ]:
# Inline playback
vmin, vmax = np.percentile(video, [1, 99])

fig, ax = plt.subplots(figsize=(7, 5))
im  = ax.imshow(video[0], cmap='inferno', vmin=vmin, vmax=vmax, animated=True)
ttl = ax.set_title('')
plt.colorbar(im, ax=ax, label='Raw counts (uint16)')
plt.tight_layout()

def _update(i):
    im.set_data(video[i])
    ttl.set_text(f'Frame {i}  (t = {i / RECORD_FPS:.3f} s)')
    return im, ttl

ani = animation.FuncAnimation(
    fig, _update, frames=len(video),
    interval=1000 / RECORD_FPS, blit=True
)
plt.close(fig)          # prevent double display
HTML(ani.to_jshtml())

## 7. Visualize

### Temporal statistics

In [ ]:
mean_img = video.mean(axis=0)
std_img  = video.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im0 = axes[0].imshow(mean_img, cmap='inferno',
                     vmin=np.percentile(mean_img, 1),
                     vmax=np.percentile(mean_img, 99))
axes[0].set_title('Temporal mean')
plt.colorbar(im0, ax=axes[0], shrink=0.8, label='counts')

im1 = axes[1].imshow(std_img, cmap='hot')
axes[1].set_title('Temporal std  (noise / motion map)')
plt.colorbar(im1, ax=axes[1], shrink=0.8, label='counts')

plt.tight_layout()
plt.show()

### Pixel time series

In [ ]:
py, px = h // 2, w // 2          # centre pixel
t = np.arange(len(video)) / RECORD_FPS

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, video[:, py, px], linewidth=0.8)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Raw counts (uint16)')
ax.set_title(f'Centre pixel ({px}, {py})')
ax.set_xlim(0, t[-1])
plt.tight_layout()
plt.show()

## 8. Disconnect

In [ ]:
cam.disconnect()
print("Disconnected.", "streaming:", cam.is_streaming)